<div style="text-align:center; padding:18px 0 6px 0"></div>

<div style="text-align:center; color:#8b949e; font-size:12px; padding-bottom:8px">Copyright © QPerfect &middot; Released under the MIT License &middot; <a href="https://qperfect.io">qperfect.io</a></div>

---

# QAOA the VQE way — server-side optimisation on MIMIQ

In `qaoa_demo.ipynb` we ran the COBYLA loop **client-side**: each
iteration submitted a fresh circuit to MIMIQ, waited, computed the
energy, picked new angles, repeated. That's a lot of round-trips.

MIMIQ has a more elegant alternative — `OptimizationExperiment` +
`conn.optimize(...)`. You hand in **one** parametric circuit (with
symbolic angles) ending in `circuit.push_expval(H, *qubits)` so the
expectation value of the cost Hamiltonian lands in a Z-register, plus
the initial parameter values and an optimiser name. The server then runs
the entire COBYLA loop in-house and returns the trajectory.

This notebook tests the idea on a **tiny hand-built QUBO** — a 4-vertex
MaxCut on a 4-cycle. The QAOA optimum here is well-known
(alternating-bit state) so we can sanity-check the server-side loop
against the textbook answer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mimiqcircuits as mc
from strasbourg_markets_demo import theme
from strasbourg_markets_demo.qubo import QUBO, bitstring_to_array
from strasbourg_markets_demo.qaoa import parametric_qaoa
from strasbourg_markets_demo.cache import mimiq_cache

theme.apply()


## 1. The toy QUBO — MaxCut on a 4-cycle

Vertices `0–1–2–3–0`, four edges. The MaxCut objective in QUBO form
(see `qubo_mappings.ipynb` §1):

$$\min_{x\in\{0,1\}^4}\; \sum_{(i,j) \in E}\bigl[-x_i - x_j + 2 x_i x_j\bigr]
\;=\; -\sum_v \deg(v)\, x_v \;+\; 2 \sum_{(i,j)\in E} x_i x_j$$

Optimal cut value = 4 (all four edges cut), achieved by alternating
configurations `0101` and `1010`. QUBO energy at those points: **−4**.


In [ ]:
n = 4
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
qubo = QUBO(Q=np.zeros((n, n)))
for v in range(n):
    qubo.add_linear(v, -2.0)              # -deg(v) · x_v  (deg = 2 on 4-cycle)
for i, j in edges:
    qubo.add_quadratic(i, j, 2.0)         # +2 · x_i x_j

# Brute-force ground state for sanity
import itertools
best = min(itertools.product([0, 1], repeat=n),
           key=lambda x: qubo.evaluate(x))
print(f"classical optimum: x = {best}    energy = {qubo.evaluate(best)}")
print(f"all minimisers:")
for x in itertools.product([0, 1], repeat=n):
    if qubo.evaluate(x) == qubo.evaluate(best):
        print(f"  {x}    energy = {qubo.evaluate(x)}")


## 2. Build the parametric QAOA ansatz with `push_expval`

`strasbourg_markets_demo.qaoa.parametric_qaoa(qubo, p)` does the
whole construction in one call:

- Defines `symengine` symbols `g0…g_{p-1}` and `b0…b_{p-1}`.
- Normalises `(h, J)` so coefficients are O(1).
- Builds the H-wall + p layers of (cost RZ/RZZ + mixer RX).
- Appends `push_expval(H_C, *qubits)` to land ⟨H_C⟩ in z-register 0.
- Returns the circuit and an `OrderedDict[Symbol, float]` of
  linear-ramp initial values — ready for `OptimizationExperiment`.

We grab the Ising offset `c` separately so we can convert
`⟨H_C⟩` energies back to QUBO units in §6.


In [ ]:
P_LAYERS = 2

_h, _J, c_offset = qubo.to_ising()    # offset only — values not reused
ansatz, init = parametric_qaoa(qubo, p=P_LAYERS)

print(f"parametric ansatz: {len(ansatz)} instructions, depth {ansatz.depth()}")
print(f"free parameters:   {ansatz.listvars()}")
print(f"num_zvars:         {ansatz.num_zvars()}  (one per ZZ term + final Add)")
print(f"Ising offset c:    {c_offset:.4f}  (carried for QUBO-unit reporting)")


## 3. Connect to MIMIQ

Standard `conn.connect()` browser-auth flow. The token cache makes the
second call instant.


In [ ]:
conn = mc.MimiqConnection(mc.QPERFECT_CLOUD)
conn.connect()
print(f"connected to {mc.QPERFECT_CLOUD}")


## 4. Configure and submit the `OptimizationExperiment`

`parametric_qaoa` already returned `init` as an `OrderedDict[Symbol,
float]` populated with linear-ramp values, so we can drop it straight
into `OptimizationExperiment`. The keys are `Symbol` objects (matching
`ansatz.listvars()`) — no string/Symbol mismatch to worry about.


In [ ]:
exp = mc.OptimizationExperiment(
    circuit=ansatz,
    initparams=init,
    optimizer="COBYLA",
    label="qaoa-server-side-toy",
    maxiters=40,
    zregister=0,                      # push_expval wrote ⟨H⟩ here
)
print(f"initparams ({exp.num_params()} symbols, linear ramp):")
for sym, val in init.items():
    print(f"  {sym} = {val:.4f}")
print(f"is_valid: {exp.is_valid()}")


In [ ]:
with mimiq_cache(conn, key="qaoa-server-toy"):
    job = conn.optimize(
        exp,
        algorithm="mps",
        bonddim=64,
        nsamples=512,
        seed=2024,
        history=True,
    )
    print(f"optimization submitted; job = {job}")
    res = conn.get_result(job)
print(f"got {type(res).__name__}: {len(res.history)} iterations recorded")


## 5. Inspect convergence and decode the optimum

`conn.get_result(job)` returns an `OptimizationResults`. The accessors:

- `get_best()` — the `OptimizationRun` of the best iteration (with
  `cost`, `parameters`, and the underlying `QCSResults`).
- `history` — list of per-iteration `OptimizationRun`s.
- `get_resultofbest()` — shortcut for the `QCSResults` of the best
  iteration (== `get_best().get_resultofbest()`).

**Note on the offset.** `qubo.to_hamiltonian()` drops the constant
$c$ from the Ising form (irrelevant to QAOA dynamics — it just shifts
all eigenvalues uniformly). So the cost $\langle H_C \rangle$ is offset
from the QUBO energy by exactly $c$:

$$\text{QUBO energy}(x) \;=\; \langle H_C \rangle(z) \;+\; c$$

For our 4-cycle MaxCut, $c = -2$. Adding it back to the convergence
trace puts everything on the QUBO scale.


In [ ]:
best_run = res.get_best()
history_runs = res.history
print(f"best cost (Hamiltonian)  = {best_run.cost:.4f}")
print(f"best params              = {dict(best_run.get_params())}")
print(f"history                  = {len(history_runs)} iterations")
print(f"Ising-conversion offset c = {c_offset:.4f}")
print(f"best QUBO energy         = best_cost + c = {best_run.cost + c_offset:.4f}")


In [ ]:
from strasbourg_markets_demo.strasbourg import place_legend_outside

# Add c_offset so the y-axis is in QUBO-energy units (matches the
# histogram column below, which uses qubo.evaluate directly).
qubo_costs = [r.cost + c_offset for r in history_runs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(qubo_costs)), qubo_costs, "o-",
        color=theme.PALETTE["cyan"], lw=2, ms=6,
        label="server-side COBYLA")
ax.axhline(-4.0, color=theme.PALETTE["green"], ls="--", lw=1.5,
           label="QUBO optimum (-4)")
ax.set_xlabel("iteration")
ax.set_ylabel("⟨H_C⟩ + c   (QUBO energy)")
ax.set_title(f"Server-side QAOA on MaxCut(C_4) — convergence  (c = {c_offset:.1f})")
place_legend_outside(ax)
plt.show()


In [ ]:
best_qcs = res.get_resultofbest()
hist = best_qcs.histogram()
nshots = sum(hist.values())
print(f"best-iteration sampling: {nshots} shots, {len(hist)} unique outcomes")
print()
print(f"{'bitstring':>10s}  {'count':>6s}  {'prob':>6s}  {'QUBO energy':>11s}")
print("-" * 40)
for bs, count in sorted(hist.items(), key=lambda kv: -kv[1])[:8]:
    x = bitstring_to_array(bs, n)
    e = qubo.evaluate(x)
    bs_str = bs.to01() if hasattr(bs, 'to01') else str(bs)
    print(f"{bs_str:>10s}  {count:>6d}  {count/nshots:>6.3f}  {e:>11.2f}")


## Summary

- Server-side optimisation collapses the entire QAOA hybrid loop into
  one `conn.optimize(...)` round-trip. No per-iteration round-trip cost
  and no client-side state to manage. We still wrap it in `mimiq_cache`
  so that re-running the notebook reads the `OptimizationResults` (the
  full history) straight from disk, just like the client-side loop in
  `qaoa_demo.ipynb`.
- The recipe in three lines:
  1. build a parametric `Circuit` with `symengine.Symbol`s for angles;
  2. append `circuit.push_expval(H_cost, *qubits)` so ⟨H⟩ lands in a Z-register;
  3. wrap in `mc.OptimizationExperiment(...)` and submit via `conn.optimize`.
- Compared to the client-side loop in `qaoa_demo.ipynb`, you trade direct
  control of each step for one-shot simplicity. Both work; pick the
  client-side path when you need custom logic per iteration (CVaR,
  branching, restarts) and the server-side path when the loop is
  textbook.
- The QUBO → Ising reduction drops a constant `c` (it doesn't affect
  the argmin and would only shift every eigenvalue equally). Adding it
  back to the displayed cost — `⟨H_C⟩ + c` — gives the QUBO energy
  directly, which is what the histogram column uses via `qubo.evaluate`.


## Appendix — flush this notebook's cache

This notebook stores only entries under the `qaoa-server-` prefix
(currently a single `qaoa-server-toy` `OptimizationResults`). The cell
below — guarded by `if False:` — wipes them and only them. The
`qaoa-tsp4-…` and `fidsweep-…` caches owned by other notebooks are
untouched.


In [ ]:
if False:
    from strasbourg_markets_demo.cache import clear_cache
    n = clear_cache(prefix="qaoa-server-")
    print(f"removed {n} cached entr{'y' if n == 1 else 'ies'} for this notebook")



---

<div style="text-align:center; padding:18px 0 6px 0"></div>

<div style="text-align:center; color:#8b949e; font-size:12px; padding-bottom:8px">Copyright © QPerfect &middot; Released under the MIT License &middot; <a href="https://qperfect.io">qperfect.io</a></div>